# Notebook 02: Data Preparation and Integration

**Project:** Pharmacogenomics Machine Learning

**Author:** Sofia Muñoz

**Purpose:** This notebook prepares the raw ClinPGx datasets for machine learning by analyzing dataset relationships, determining merge strategies, cleaning the data, engineering features, and producing a final master dataset suitable for model development.

**Datasets:**
- Clinical Variants
- Variant Drug Annotations
- Variant Phenotype Annotations
- Variant Functional Assay Annotations

## Objectives

By the end of this notebook, I will:
- Identify relationships among all four datasets.
- Determine scientifically appropriate merge keys.
- Validate candidate merge keys.
- Develop a documented merge strategy.
- Clean and standardize each dataset.
- Integrate the datasets into a single master table.
- Engineer features for machine learning.
- Export a processed dataset for model development.

## Table of Contents

1. Import Libraries
2. Load Raw Datasets
3. Standardize Column Names
4. Dataset Schema Harmonization
5. Dataset Relationship Analysis
6. Candidate Merge Keys
7. Merge Strategy
8. Data Cleaning
9. Standardization
10. Missing Values
11. Duplicate Analysis
12. Dataset Integration
13. Feature Engineering
14. Final Quality Checks
15. Export Processed Dataset
16. Reflection

### 1. Import Libraries

In [89]:
# Import the pandas library for reading, manipulating, and analyzing tabular data.
import pandas as pd

# Import NumPy for numerical operations.
import numpy as np

# Import Matplotlib for creating graphs and visualizations.
import matplotlib.pyplot as plt

# Import Path from pathlib to build operating system-independent file paths.
from pathlib import Path

### 2. Load Raw Datasets

In [90]:
# Load the Clinical Variants dataset.
clinical_raw = pd.read_csv(
    "../data/raw/clinicalVariants.tsv",
    sep="\t"
)
# Load the Variant Drug Annotations dataset.
drug_raw = pd.read_csv(
    "../data/raw/var_drug_ann.tsv",
    sep="\t"
)
# Load the Variant Phenotype Annotations dataset.
phenotype_raw = pd.read_csv(
    "../data/raw/var_pheno_ann.tsv",
    sep="\t"
)
# Load the Variant Functional Assays Annotations dataset.
functional_raw = pd.read_csv(
    "../data/raw/var_fa_ann.tsv",
    sep="\t"
)

In [91]:
# Verify the files loaded
print("Clinical Variants:", clinical_raw.shape)
print("Drug Annotations:", drug_raw.shape)
print("Phenotype Annotations:", phenotype_raw.shape)
print("Functional Assays:", functional_raw.shape)

Clinical Variants: (5190, 6)
Drug Annotations: (12975, 22)
Phenotype Annotations: (14490, 25)
Functional Assays: (2153, 23)


In [92]:
# Create working copies of each dataset.
# All preprocessing will be performed on these copies, preserving the original raw datasets for reference.

clinical = clinical_raw.copy()
drug = drug_raw.copy()
phenotype = phenotype_raw.copy()
functional = functional_raw.copy()

In [93]:
# Verify the copies
print(clinical.shape == clinical_raw.shape)
print(drug.shape == drug_raw.shape)
print(phenotype.shape == phenotype_raw.shape)
print(functional.shape == functional_raw.shape)

True
True
True
True


### 3. Standardize Column Names

The column names of the working datasets are standardized to a consistent
snake_case convention. This improves readability and prevents inconsistencies
when referring to columns throughout the preprocessing and integration
pipeline.

Only the working copies are modified. The original raw DataFrames remain
unchanged.

In [94]:
# Standardize column names across all working datasets.
# The transformation:
#   1. Removes leading and trailing whitespace.
#   2. Converts all characters to lowercase.
#   3. Replaces spaces with underscores.

def standardize_column_names(df):
    """Return a DataFrame with standardized snake_case column names."""
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )
    return df

# Apply the standardization function to each working dataset.
clinical = standardize_column_names(clinical)
drug = standardize_column_names(drug)
phenotype = standardize_column_names(phenotype)
functional = standardize_column_names(functional)

In [95]:
# Display the standardized column names for each dataset.

print("Clinical Variants:")
print(clinical.columns.tolist())

print("\nVariant Drug Annotations:")
print(drug.columns.tolist())

print("\nVariant Phenotype Annotations:")
print(phenotype.columns.tolist())

print("\nVariant Functional Assay Annotations:")
print(functional.columns.tolist())

Clinical Variants:
['variant', 'gene', 'type', 'level_of_evidence', 'chemicals', 'phenotypes']

Variant Drug Annotations:
['variant_annotation_id', 'variant_haplotypes', 'gene', 'drug_s', 'pmid', 'phenotype_category', 'significance', 'notes', 'sentence', 'alleles', 'specialty_population', 'metabolizer_types', 'isplural', 'is_is_not_associated', 'direction_of_effect', 'pd_pk_terms', 'multiple_drugs_and_or', 'population_types', 'population_phenotypes_or_diseases', 'multiple_phenotypes_or_diseases_and_or', 'comparison_allele_s_or_genotype_s', 'comparison_metabolizer_types']

Variant Phenotype Annotations:
['variant_annotation_id', 'variant_haplotypes', 'gene', 'drug_s', 'pmid', 'phenotype_category', 'significance', 'notes', 'sentence', 'alleles', 'specialty_population', 'metabolizer_types', 'isplural', 'is_is_not_associated', 'direction_of_effect', 'side_effect_efficacy_other', 'phenotype', 'multiple_phenotypes_and_or', 'when_treated_with_exposed_to_when_assayed_with', 'multiple_drugs_and

### 4. Dataset Schema Harmonization

Standardizing column names improves consistency within the computational
pipeline, but similarly named columns are not assumed to represent the
same biological concept.

Potentially equivalent columns are therefore evaluated based on their
definitions, contents, data types, missingness, and biological meaning
before being harmonized for dataset integration.

In [96]:
# Compare example values from the potentially equivalent variant columns.

print("Clinical Variants:")
print(clinical["variant"].dropna().head(10))

print("\nVariant Drug Annotations:")
print(drug["variant_haplotypes"].dropna().head(10))

print("\nVariant Phenotype Annotations:")
print(phenotype["variant_haplotypes"].dropna().head(10))

print("\nVariant Functional Assay Annotations:")
print(functional["variant_haplotypes"].dropna().head(10))

Clinical Variants:
0                        CYP2C9*1, CYP2C9*3, CYP2C9*13
1                                           rs17376848
2                                            rs2297595
3                                            rs1801265
4                      CYP2C19*1, CYP2C19*2, CYP2C19*3
5    CYP2C9*1, CYP2C9*2, CYP2C9*3, CYP2C9*5, CYP2C9...
6                                            rs1801160
7                                            rs1801159
8    UGT1A1*1, UGT1A1*6, UGT1A1*28, UGT1A1*36, UGT1...
9            TPMT*1, TPMT*2, TPMT*3A, TPMT*3B, TPMT*3C
Name: variant, dtype: str

Variant Drug Annotations:
0     CYP3A4*1, CYP3A4*17
1               rs2909451
2                rs706795
3              rs16918842
4      CYP2C9*1, CYP2C9*3
5               rs2285676
6               CYP2C9*11
7                rs163184
8     CYP2B6*1, CYP2B6*18
9    CYP2C19*1, CYP2C19*2
Name: variant_haplotypes, dtype: str

Variant Phenotype Annotations:
0                    HLA-B*35:08
1               

In [97]:
print(clinical["variant"].dtype)
print(drug["variant_haplotypes"].dtype)
print(phenotype["variant_haplotypes"].dtype)
print(functional["variant_haplotypes"].dtype)

str
str
str
str


In [98]:
print(
    "Clinical:",
    clinical["variant"].isna().mean()
)
print(
    "Drug:",
    drug["variant_haplotypes"].isna().mean()
)
print(
    "Phenotype:",
    phenotype["variant_haplotypes"].isna().mean()
)
print(
    "Functional:",
    functional["variant_haplotypes"].isna().mean()
)

Clinical: 0.0
Drug: 0.0
Phenotype: 0.0
Functional: 0.0


### 5. Dataset Relationship Analysis

#### 5.1 Biological Roles of Each Dataset

| Dataset | What one row represents | Primary purpose |
| :--- | :--- | :--- |
| **Clinical Variants** | One curated clinical pharmacogenomic association describing how a specific genetic variant (or genotype/haplotype) influences a drug-related outcome (such as efficacy, toxicity, dosage, or metabolism) based on published clinical evidence. | Summarizes clinically actionable pharmacogenomic knowledge and recommendations for healthcare decision-making. |
| **Variant Drug Annotations** | One evidence record describing the relationship between a specific genetic variant and a particular drug, including the reported pharmacogenomic effect from an individual study or publication. Multiple rows may exist for the same variant because different drugs, studies, or evidence sources can be associated with it. | Provides detailed evidence linking genetic variants to drug response and serves as the primary source of variant–drug relationships. |
| **Variant Phenotype Annotations** | One evidence record describing how a specific genetic variant is associated with an observed phenotype (for example, altered metabolism, treatment response, or adverse drug reaction) reported in a publication. | Connects genetic variants with observed pharmacogenomic phenotypes that may explain differences in medication response. |
| **Variant Functional Assay Annotations** | One laboratory experimental result measuring the functional impact of a specific genetic variant on gene or protein activity. These data come from experimental assays rather than clinical observations. | Provides biological evidence about how variants affect molecular function, supporting interpretation of clinical and pharmacogenomic findings. |

#### 5.2 Candidate Shared Columns

| Column | Clinical | Drug | Phenotype | Functional |
| :--- | :---: | :---: | :---: | :---: |
| Variant Annotation ID | X | ✓ | ✓ | ✓ |
| Gene | ✓ | ✓ | ✓ | ✓ |
| Variant | ✓ | ✓ | ✓ | ✓ |
| Alleles | X | ✓ | ✓ | ✓ |
| Drug | ✓ | ✓ | ✓ | ✓ |
| PMID | X | ✓ | ✓ | ✓ |

#### 5.3 Candidate Merge Key Evaluation

##### Candidate Key 1: Variant Annotation ID

Variant Annotation ID is evaluated first because it appears in the three variant annotation datasets and is intended to uniquely identify a variant/drug annotation.

In [99]:
# Determine whether Variant Annotation ID exists in each dataset.

candidate_key1 = "variant_annotation_id"

datasets = {
    "Clinical Variants": clinical,
    "Variant Drug Annotations": drug,
    "Variant Phenotype Annotations": phenotype,
    "Variant Functional Assay Annotations": functional
}
for name, df in datasets.items():
    print(f"{name}: {candidate_key1 in df.columns}")

Clinical Variants: False
Variant Drug Annotations: True
Variant Phenotype Annotations: True
Variant Functional Assay Annotations: True


*Observation*

Variant Annotation ID is present in the Variant Drug Annotations, Variant Phenotype Annotations, and Variant Functional Assay Annotations datasets but is absent from the Clinical Variants dataset.

This suggests that it may serve as the primary integration key among the variant annotation datasets but cannot directly link the Clinical Variants dataset.

In [100]:
for name, df in datasets.items():
    if candidate_key1 in df.columns:
        missing = df[candidate_key1].isna().sum()
        print(f"{name}: {missing} missing values")

Variant Drug Annotations: 0 missing values
Variant Phenotype Annotations: 0 missing values
Variant Functional Assay Annotations: 0 missing values


In [101]:
for name, df in datasets.items():
    if candidate_key1 in df.columns:
        print(name)
        print("Unique:",
              df[candidate_key1].is_unique)
        print()

Variant Drug Annotations
Unique: True

Variant Phenotype Annotations
Unique: True

Variant Functional Assay Annotations
Unique: True



In [102]:
summary1 = pd.DataFrame({
    "Question": [
        "Present in datasets?",
        "Unique?",
        "Missing values?",
        "Represents biology?",
        "Merge candidate?"
    ],
    "Answer": [
        "All but Clinical Variants",
        "Yes",
        "No",
        "Yes",
        "Yes"
    ]})
summary1

,Question,Answer
0,Present in datasets?,All but Clinical Variants
1,Unique?,Yes
2,Missing values?,No
3,Represents biology?,Yes
4,Merge candidate?,Yes


##### Candidate Key 2: Gene

In [103]:
# Determine whether Variant Annotation ID exists in each dataset.

candidate_key2 = "gene"

datasets = {
    "Clinical Variants": clinical,
    "Variant Drug Annotations": drug,
    "Variant Phenotype Annotations": phenotype,
    "Variant Functional Assay Annotations": functional
}
for name, df in datasets.items():
    print(f"{name}: {candidate_key2 in df.columns}")

Clinical Variants: True
Variant Drug Annotations: True
Variant Phenotype Annotations: True
Variant Functional Assay Annotations: True


In [104]:
for name, df in datasets.items():
    if candidate_key2 in df.columns:
        missing = df[candidate_key2].isna().sum()
        print(f"{name}: {missing} missing values")

Clinical Variants: 254 missing values
Variant Drug Annotations: 340 missing values
Variant Phenotype Annotations: 416 missing values
Variant Functional Assay Annotations: 50 missing values


In [105]:
for name, df in datasets.items():
    if candidate_key2 in df.columns:
        print(name)
        print("Unique:",
              df[candidate_key2].is_unique)
        print()

Clinical Variants
Unique: False

Variant Drug Annotations
Unique: False

Variant Phenotype Annotations
Unique: False

Variant Functional Assay Annotations
Unique: False



In [106]:
summary2 = pd.DataFrame({
    "Question": [
        "Present in datasets?",
        "Unique?",
        "Missing values?",
        "Represents biology?",
        "Merge candidate?"
    ],
    "Answer": [
        "All",
        "No",
        "Yes",
        "Yes",
        "..."
    ]})
summary2

,Question,Answer
0,Present in datasets?,All
1,Unique?,No
2,Missing values?,Yes
3,Represents biology?,Yes
4,Merge candidate?,...


##### Candidate Key 3: Variant

In [107]:
# Determine whether Variant exists in each dataset.

candidate_key3 = "variant"

datasets = {
    "Clinical Variants": clinical,
    "Variant Drug Annotations": drug,
    "Variant Phenotype Annotations": phenotype,
    "Variant Functional Assay Annotations": functional
}
for name, df in datasets.items():
    print(f"{name}: {candidate_key3 in df.columns}")

Clinical Variants: True
Variant Drug Annotations: False
Variant Phenotype Annotations: False
Variant Functional Assay Annotations: False


In [108]:
for name, df in datasets.items():
    if candidate_key3 in df.columns:
        missing = df[candidate_key3].isna().sum()
        print(f"{name}: {missing} missing values")

Clinical Variants: 0 missing values


In [109]:
for name, df in datasets.items():
    if candidate_key3 in df.columns:
        print(name)
        print("Unique:",
              df[candidate_key3].is_unique)
        print()

Clinical Variants
Unique: False



In [110]:
summary3 = pd.DataFrame({
    "Question": [
        "Present in datasets?",
        "Unique?",
        "Missing values?",
        "Represents biology?",
        "Merge candidate?"
    ],
    "Answer": [
        "All but Clinical Variants",
        "Yes",
        "No",
        "Yes",
        "Yes"
    ]})
summary3

,Question,Answer
0,Present in datasets?,All but Clinical Variants
1,Unique?,Yes
2,Missing values?,No
3,Represents biology?,Yes
4,Merge candidate?,Yes
